In [0]:
from pyspark.sql import functions as F

In [0]:
adls_path = "abfss://adb-container@qcadlsbatch10.dfs.core.windows.net"
file_path = f"{adls_path}/raw_data/selected-price-indexes-june-2026.csv"

prices_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(file_path)
)

prices_df.display()

In [0]:
prices_df = (
    prices_df
    .withColumn("Period", F.col("Period").cast("int"))
    .withColumn("Data_value", F.col("Data_value").try_cast("double"))
    .withColumn("Data_value", F.col("Data_value").try_cast("int"))
    .groupBy("Period")
    .agg(F.sum("Data_value").alias("total_data_value"))
) 

prices_df.display()

In [0]:
target_path = f"{adls_path}/curated_data/selected_price_indexes_total"
prices_df.write.format("csv").mode("overwrite").option("header", True).save(target_path)

In [0]:
target_path = f"{adls_path}/curated_data/selected_price_indexes_total_parquet"
prices_df.write.format("parquet").mode("overwrite").save(target_path)

In [0]:
p_df = spark.read.format("parquet").load(f"{adls_path}/curated_data/selected_price_indexes_total_parquet")
p_df.display()

In [0]:
output_df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(f"{adls_path}/raw_data/output_csv_full.csv")
)

output_df.display()

In [0]:
output_df.write.format("csv").option("header", True).mode("overwrite").save(f"{adls_path}/curated_data/output_csv_full")
output_df.write.format("parquet").mode("overwrite").save(f"{adls_path}/curated_data/output_parquet_full")